In [ ]:
# Basis
from pathlib import Path
import geopandas as gpd
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
from shapely.geometry import Point, Polygon
import contextily as cx
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
import shutil
import os

In [ ]:
# and from hydrolib-core
from hydrolib.core.dimr.models import DIMR, FMComponent
from hydrolib.core.dflowfm.inifield.models import IniFieldModel, DiskOnlyFileModel
from hydrolib.core.dflowfm.onedfield.models import OneDFieldModel
from hydrolib.core.dflowfm.structure.models import StructureModel
from hydrolib.core.dflowfm.crosssection.models import CrossLocModel, CrossDefModel
from hydrolib.core.dflowfm.ext.models import ExtModel
from hydrolib.core.dflowfm.mdu.models import FMModel
from hydrolib.core.dflowfm.friction.models import FrictionModel
from hydrolib.core.dflowfm.obs.models import ObservationPointModel
from hydrolib.core.dflowfm.storagenode.models import StorageNodeModel

In [ ]:
from hydrolib.dhydamo.core.hydamo import HyDAMO
from hydrolib.dhydamo.converters.df2hydrolibmodel import Df2HydrolibModel
from hydrolib.dhydamo.geometry import mesh
from hydrolib.dhydamo.core.drr import DRRModel
from hydrolib.dhydamo.core.drtc import DRTCModel
from hydrolib.dhydamo.io.dimrwriter import DIMRWriter
from hydrolib.dhydamo.io.drrwriter import DRRWriter
from hydrolib.dhydamo.geometry.viz import plot_network
from meshkernel.py_structures import DeleteMeshOption

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
# Hier begint de functie
def generate_rr_model(dir_data, dir_output, selectie_gebied, scenario, seizoen, start_date, end_date, restart_in):
    
    dir_gebied = Path(dir_data, "rr_input_area_scenario", f"gebied_{selectie_gebied}")
    dir_scenarios = Path(dir_data, "rr_input_scenarios")
    dir_output_gebied = Path(dir_output, f"gebied_{selectie_gebied}")

    gebied = gpd.read_file(dir_gebied / f"gebied.gpkg", layer=f"gebied")
    watergang = gpd.read_file(dir_gebied / f"watergang.gpkg", layer=f"watergang")
    afwateringseenheden = gpd.read_file(dir_gebied / f"afwateringseenheden.gpkg", layer=f"afwateringseenheden")

    #Initialize RR model
    drrmodel = DRRModel()

    #Read modelinput
    df_unpaved = pd.read_csv(dir_gebied / scenario / f"df_unpaved_{seizoen}.csv")
    df_ernst = pd.read_csv(dir_gebied / scenario / f"df_ernst_{seizoen}.csv")

    #add unpaved nodes
    #####Hier staat nu nog alleen zomer####
    for unpaved_dict in df_unpaved.set_index("code").drop(columns="boundary_waterlevel").to_dict("records"):
        drrmodel.unpaved.add_unpaved(**unpaved_dict)
    #add ernst definitions
    for ernst_dict in df_ernst.set_index("code").to_dict("records"):
        drrmodel.unpaved.add_ernst_def(**ernst_dict)

    # select project extend
    path_extent_project_area = dir_gebied / "gebied.gpkg"
    hydamo = HyDAMO(extent_file=path_extent_project_area)

    #read branches
    path_watergangen = Path(dir_gebied, "watergang.gpkg")
    hydamo.branches.read_gpkg_layer(path_watergangen, layer_name="watergang", index_col="code")

    # read catchments
    path_afwateringseenheden = Path(dir_gebied, "afwateringseenheden.gpkg")
    hydamo.catchments.read_gpkg_layer(path_afwateringseenheden, layer_name="afwateringseenheden", index_col="code", check_geotype=False)

    laterals_gdf = gpd.GeoDataFrame(df_unpaved, geometry=gpd.points_from_xy(df_unpaved.px-10, df_unpaved.py-10), crs=28992)
    laterals_gdf["rr_node"] = laterals_gdf["code"]
    laterals_gdf["code"] = laterals_gdf["boundary_node"]
    laterals_gdf["globalid"] = laterals_gdf["code"]
    # select all relevant columns for laterale_knoop
    laterals_gdf_selection = laterals_gdf[["code", "rr_node", "globalid", "geometry"]].set_index("code").copy()
    laterals_gdf_selection.to_file(dir_data / "laterale_knoop.gpkg", layer="laterale_knoop", driver="GPKG")

    # read laterals and match them to the catchments
    hydamo.laterals.read_gpkg_layer(dir_data / "laterale_knoop.gpkg", layer_name="laterale_knoop")
    hydamo.laterals.snap_to_branch(hydamo.branches, snap_method="overal", maxdist=5000)
    hydamo.catchments['boundary_node'] = hydamo.catchments['lateraleknoopid'].copy()
    
    #generate RR boundaries
    drrmodel.external_forcings.io.boundary_from_input(
        hydamo.laterals, 
        hydamo.catchments, 
        drrmodel, 
    )

    # External forcings
    # 3 types of extrenal forcings need to be provided: seepage, precipitation and evaporation
    # read forcings
    seepage_file = str(dir_scenarios / "scenarios" / scenario / "kwel_per_RR_knoop.csv")
    precip_file = str(dir_gebied / "meteo" / "METEO_NEERSLAG.BUI")
    evap_file = str(dir_gebied / "meteo" / "METEO_VERDAMPING.EVP")

    seepage_df = pd.read_csv(seepage_file, index_col=0, parse_dates=True)[start_date:end_date]
    [drrmodel.external_forcings.add_seepage(*sep) for sep in seepage_df.items()]
    drrmodel.external_forcings.io.precip_from_input(hydamo.catchments, precip_folder=None, precip_file=precip_file)
    drrmodel.external_forcings.io.evap_from_input(hydamo.catchments, evap_folder=None, evap_file=evap_file)

    #Add the main parameters
    drrmodel.d3b_parameters["Timestepsize"] = 300
    drrmodel.d3b_parameters["StartTime"] = "'" + pd.to_datetime(start_date).strftime('%Y/%m/%d;%H:%M:%S') + "'"  # should be equal to refdate for D-HYDRO
    drrmodel.d3b_parameters["EndTime"] = "'" + pd.to_datetime(end_date).strftime('%Y/%m/%d;%H:%M:%S') + "'"
    drrmodel.d3b_parameters["RestartIn"] = restart_in
    drrmodel.d3b_parameters["RestartOut"] = 1
    # drrmodel.d3b_parameters["RestartFileNamePrefix"] = "Test"
    drrmodel.d3b_parameters["OutputAtTimestep"] = 12
    drrmodel.d3b_parameters["UnsaturatedZone"] = 1
    drrmodel.d3b_parameters["UnpavedPercolationLikeSobek213"] = -1
    drrmodel.d3b_parameters["VolumeCheckFactorToCF"] = 100000

    #Convert laterals to RR boundaries
    hydamo.external_forcings.convert.laterals(
        hydamo.laterals,
        # overflows=hydamo.overflows,
        # greenhouse_laterals=hydamo.greenhouse_laterals,
        lateral_discharges=None,
        rr_boundaries=drrmodel.external_forcings.boundary_nodes
    )

    #Writing the model
    # overwrite output-path to write the models
    # model_name = 
    output_path = dir_output_gebied / scenario / f"rr_model__{pd.to_datetime(start_date).strftime('%Y_%m_%d')}__{pd.to_datetime(end_date).strftime('%Y_%m_%d')}"

    if output_path.exists():
        shutil.rmtree(output_path)
    output_path.mkdir(parents=True)

    rr_writer = DRRWriter(drrmodel, output_dir=output_path)
    rr_writer.write_all()

    # rewrite boundary conditions
    filepath = Path(output_path, "rr", "BoundaryConditions.bc")
    header = {"fileVersion": "1.01", "fileType": "boundConds"}
    with open(filepath, "w") as f:
        rr_writer._write_dict(f, header, "General", "\n")
        for _, dct in rr_writer.rrmodel.external_forcings.boundary_nodes.items():
            temp = {
                "name": "" + dct["id"],
                "function": "constant",
                "quantity": "water_level",
                "unit": "m",
            }
            rr_writer._write_dict(f, temp, "Boundary", f"    {laterals_gdf.set_index('code').loc[dct['id'],'boundary_waterlevel']}\n\n")
        if any(rr_writer.rrmodel.paved.pav_nodes):
            temp = {
                "name": "WWTP_BND",
                "function": "constant",
                "quantity": "water_level",
                "unit": "m",
            }
            rr_writer._write_dict(f, temp, "Boundary", "    0\n\n")

    #dir_assets = Path("..\\src\\wrij_rr_unpaved_methode\\assets")
    
    shutil.copy(
        dir_assets / "dimr_config.xml",
        output_path / "dimr_config.xml"
    )
    
    shutil.copy(
        dir_assets / "run.bat",
        output_path / "run.bat"
    )
    return drrmodel, hydamo

Define in- and output paths

In [ ]:
# selectie_gebied = 0 # Oude IJssel
# selectie_gebied = 1 # West
# selectie_gebied = 2 # Centraal
# selectie_gebied = 3 # Oost

selectie_gebieden = [3]
# selectie_gebieden = [1, 2]

scenarios = ["REF", "SCEN"]

start_date = "2010-4-1"
end_date = "2018-12-31"
# end_date = "2010-4-11"

# path to the package containing the dummy-data
dir_data = Path("..\\..\\WRIJ_RR_Unpaved_methode_02_input")
dir_output = Path("..\\..\\WRIJ_RR_Unpaved_methode_03_modellen\\oude_ijssel")
dir_assets = Path("..\\src\\wrij_rr_unpaved_methode\\assets")

In [ ]:
date_range = pd.date_range(start_date, end_date, freq="6MS")
# date_range = pd.date_range(start_date, end_date, freq="2D")

simulaties_total = pd.DataFrame()

for selectie_gebied in selectie_gebieden:
    for scenario in scenarios:
        simulaties = pd.DataFrame()
        
        simulaties["start_date"] = date_range
        simulaties["end_date"] = simulaties["start_date"].shift(-1)
        simulaties.loc[simulaties.index[-1],"end_date"] = pd.to_datetime(end_date)
        simulaties["seizoen"] = ["zomer", "winter"]*int(len(date_range)/2)
        simulaties["scenario"] = scenario
        simulaties["gebied"] = selectie_gebied
        simulaties["restart_in"] = [0] + [1] * len(simulaties.index[1:])

        simulaties["model_name"] = simulaties.apply(lambda x: f"rr_model__{pd.to_datetime(x.start_date).strftime('%Y_%m_%d')}__{pd.to_datetime(x.end_date).strftime('%Y_%m_%d')}", axis=1)

        simulaties_total = pd.concat([simulaties_total, simulaties])

In [ ]:
simulaties_total

In [ ]:
for index, simulatie in simulaties_total.iterrows():
    display(str(simulatie.gebied) + " - " + simulatie.scenario + " - " + simulatie.model_name)
    drr_model, hydamo = generate_rr_model(dir_data, dir_output, simulatie.gebied, simulatie.scenario, simulatie.seizoen, simulatie.start_date, simulatie.end_date, simulatie.restart_in)